In [1]:
import json

with open("data/lvis_v1_train.json") as f:
    data = json.load(f)

images = data['images']
annotations = data['annotations']
categories = data['categories']

In [2]:
from collections import defaultdict

img_to_cats = defaultdict(list)

for ann in annotations:
    img_to_cats[ann['image_id']].append(ann['category_id'])

In [3]:
cat_to_images = defaultdict(set)

for img_id, cats in img_to_cats.items():
    for cat in cats:
        cat_to_images[cat].add(img_id)

In [4]:
TARGET = 15744
selected_ids = set()

# Step 1: balanced base
for cat, imgs in cat_to_images.items():
    for img_id in list(imgs)[:5]:   # small per-category sampling
        selected_ids.add(img_id)

# Step 2: fill remaining globally
for img in images:
    if len(selected_ids) >= TARGET:
        break
    selected_ids.add(img['id'])

In [5]:
image_dict = {img['id']: img for img in images}

selected_image_list = [
    image_dict[img_id] for img_id in selected_ids
]

In [6]:
selected_ids = set(img['id'] for img in selected_image_list)

In [7]:
selected_image_list

[{'flickr_url': 'http://farm3.staticflickr.com/2397/2418031840_c11907ef0b_z.jpg',
  'id': 294912,
  'neg_category_ids': [806, 1026, 455, 855, 1171, 129, 484, 264, 1197],
  'not_exhaustive_category_ids': [],
  'width': 480,
  'license': 3,
  'coco_url': 'http://images.cocodataset.org/train2017/000000294912.jpg',
  'date_captured': '2013-11-22 21:15:55',
  'height': 640},
 {'flickr_url': 'http://farm8.staticflickr.com/7187/6967031859_5f08387bde_z.jpg',
  'id': 262145,
  'neg_category_ids': [50, 691, 633, 1147, 672, 954, 847],
  'not_exhaustive_category_ids': [947, 838, 217, 514, 342],
  'width': 640,
  'license': 2,
  'coco_url': 'http://images.cocodataset.org/train2017/000000262145.jpg',
  'date_captured': '2013-11-20 02:07:55',
  'height': 427},
 {'flickr_url': 'http://farm6.staticflickr.com/5090/5341741494_1f653cdb80_z.jpg',
  'id': 262146,
  'neg_category_ids': [254, 520, 328, 1151, 742, 523, 795, 821, 617, 578],
  'not_exhaustive_category_ids': [],
  'width': 480,
  'license': 1,
  

In [8]:
categories

[{'name': 'aerosol_can',
  'instance_count': 109,
  'def': 'a dispenser that holds a substance under pressure',
  'synonyms': ['aerosol_can', 'spray_can'],
  'image_count': 64,
  'id': 1,
  'frequency': 'c',
  'synset': 'aerosol.n.02'},
 {'name': 'air_conditioner',
  'instance_count': 1081,
  'def': 'a machine that keeps air cool and dry',
  'synonyms': ['air_conditioner'],
  'image_count': 364,
  'id': 2,
  'frequency': 'f',
  'synset': 'air_conditioner.n.01'},
 {'name': 'airplane',
  'instance_count': 3720,
  'def': 'an aircraft that has a fixed wing and is powered by propellers or jets',
  'synonyms': ['airplane', 'aeroplane'],
  'image_count': 1911,
  'id': 3,
  'frequency': 'f',
  'synset': 'airplane.n.01'},
 {'name': 'alarm_clock',
  'instance_count': 158,
  'def': 'a clock that wakes a sleeper at some preset time',
  'synonyms': ['alarm_clock'],
  'image_count': 149,
  'id': 4,
  'frequency': 'f',
  'synset': 'alarm_clock.n.01'},
 {'name': 'alcohol',
  'instance_count': 207,
  '

In [9]:
import os
import shutil
import requests
from tqdm import tqdm
import json

SOURCE_ROOT = "B:/CV Project"   # must contain train2017 / val2017
DEST_ROOT = "B:/CV Project/lvis_16k_dataset"

IMG_DIR = os.path.join(DEST_ROOT, "images")
os.makedirs(IMG_DIR, exist_ok=True)

In [10]:
def get_file_name(img):
    return img['coco_url'].split("images.cocodataset.org/")[-1]


def download_image(url, save_path, retries=3):
    for _ in range(retries):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(save_path, "wb") as f:
                    f.write(r.content)
                return True
        except:
            pass
    return False

In [11]:
selected_ids = set(img['id'] for img in selected_image_list)

ann_map = {}
for ann in annotations:
    if ann['image_id'] in selected_ids:
        ann_map.setdefault(ann['image_id'], []).append(ann)

In [12]:
valid_images = []
valid_annotations = []

missing_count = 0
downloaded = 0
copied = 0
skipped_duplicates = 0

seen_files = set()

for img in tqdm(selected_image_list):

    file_name = get_file_name(img)
    base_name = os.path.basename(file_name)

    src_path = os.path.join(SOURCE_ROOT, file_name)
    dst_path = os.path.join(IMG_DIR, base_name)

    # جلوگیری duplicate copy
    if base_name in seen_files:
        skipped_duplicates += 1
        continue

    success = False

    # ✅ Case 1: exists locally
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        copied += 1
        success = True

    # 🌐 Case 2: download
    else:
        success = download_image(img['coco_url'], dst_path)
        if success:
            downloaded += 1
        else:
            missing_count += 1

    # ✅ Only keep valid entries
    if success:
        seen_files.add(base_name)

        img_copy = img.copy()
        img_copy['file_name'] = base_name  # fix path
        valid_images.append(img_copy)

        if img['id'] in ann_map:
            valid_annotations.extend(ann_map[img['id']])

100%|██████████████████████████████████████████████████████████████████████████| 15744/15744 [00:08<00:00, 1758.14it/s]


In [13]:
final_dataset = {
    "images": valid_images,
    "annotations": valid_annotations,
    "categories": categories
}

with open(os.path.join(DEST_ROOT, "annotations.json"), "w") as f:
    json.dump(final_dataset, f)

In [15]:
print("========== DATASET SUMMARY ==========")
print(f"Target images       : {len(selected_image_list)}")
print(f"Final images        : {len(valid_images)}")
print(f"Copied              : {copied}")
print(f"Downloaded          : {downloaded}")
print(f"Missing (failed)    : {missing_count}")
print(f"Duplicates skipped  : {skipped_duplicates}")

========== DATASET SUMMARY ==========
Target images       : 15744
Final images        : 15744
Copied              : 15744
Downloaded          : 0
Missing (failed)    : 0
Duplicates skipped  : 0
